# Benchmark: new vs. old vector-classification pipeline

Scores the **new** `rastervec` Vector-Classification + OCR pipeline against the
**old** archive/legacy pipeline on the same ground truth and the same metrics:

- **Current (new)**: `rastervec`'s own `Pipeline.STAGES` chain (layer/color
  separation -> 12-step Vector_Classification -> FAST text detect -> PaddleOCR),
  via `pipeline.run_page_context`.
- **Archive (legacy / old)**: `archive/raster_parser/main_pipeline_extract.extract`,
  run completely unmodified via `Evaluation/Evaluate/legacy_adapter.py`.

### Dataset

`collect_dataset(DATASET_ROOT)` recursively walks one directory tree for both
`.pdf` files and label-sidecar `.json` files and pairs them into a flat
`(pdf, page)` dataset. A tree may mix labelled and unlabelled PDFs freely.

### Ground truth — two sources, scored separately

- **auto** — `auto_label_pdf`, derived from the PDF's own native text,
  independent of either pipeline. Always generated for every benchmarked page.
- **manual** — human-entered `LabelEntry`s from a `manual_label.py` sidecar
  `.json` discovered in the tree, attached to the pages they name.

`split_labelset_by_source` keeps the two apart: every pipeline is scored once
against auto labels and once against manual labels, so `current/auto`,
`current/manual`, `legacy/auto`, `legacy/manual` each get their own aggregate
and chart series.

### Output

- **Per-page** evaluation reports and per-page stage timing → `RESULTS_TXT`
  (current) / `<stem>_legacy.txt` (legacy). Not printed.
- **Printed**: only the aggregated accuracy metrics, the aggregated per-stage +
  per-page timing distribution (min / Q1 / median / mean / Q3 / max), and the
  charts.
- **Reconstruction PDFs** (when `RECONSTRUCT_DIR` is set): each page writes
  `<stem>_p<N>_groundtruth.pdf` / `_current.pdf` / `_legacy.pdf` via
  `renderer.render_reconstructed_pdf`.

This is a **sanity/regression check**, not a real A/B: the new pipeline is
expected to score at least as well as the old one.


In [ ]:
import math
import random
import sys
import tempfile
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "benchmark_vector_classification.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pymupdf as fitz
from tqdm import tqdm

from rastervec.Evaluation.Conversion.conversion import convert_page_to_vector_text
from rastervec.Evaluation.Evaluate.evaluate import (
    DEFAULT_IOU_THRESHOLD,
    evaluate_pipeline,
    split_labelset_by_source,
)
from rastervec.Evaluation.Evaluate.benchmark import (
    aggregate_results,
    distribution_stats,
    format_report,
    format_timing_report,
    summarize_stage_timings,
)
from rastervec.Evaluation.Evaluate.legacy_adapter import (
    run_archive_pipeline,
    to_cluster_ocr_results,
    to_drawing_vectors,
)
from rastervec.Evaluation.Labelling.auto_label import auto_label_pdf
from rastervec.logging_setup import configure_logging
from rastervec.OCR.Paddle_OCR.render_ocr import MIN_RENDER_SIDE_PX
from rastervec.pipeline import Pipeline, run_page_context
from rastervec.Reader.dataset import collect_dataset
from rastervec.Reader.reader import Reader
from rastervec.renderer import (
    cluster_frame_size,
    render_reconstructed_pdf,
    render_vector_cluster,
)

configure_logging()

## Parameters

`DATASET_ROOT` is a directory tree searched **recursively** for both `.pdf`
files and label-sidecar `.json` files (the `manual_label.py` / `auto_label.py`
`LabelSet` format). `collect_dataset` pairs them into one flat `(pdf, page)`
dataset:

- A PDF a label file references → benchmarked on exactly the pages that file
  names, carrying its `source="manual"` entries.
- A PDF with no label file → benchmarked on its first `PAGES_PER_PDF` pages,
  auto labels only.

Either way, **auto** ground truth (`auto_label_pdf`, from the PDF's own native
text) is generated per page, and `split_labelset_by_source` scores auto and
manual separately (`current/auto`, `current/manual`, `legacy/auto`,
`legacy/manual`).

- `RESULTS_TXT` — per-page evaluation reports + per-page stage timing are
  written here (current pipeline) / to `<stem>_legacy.txt` (legacy). Only the
  aggregated metrics and the aggregated timing distribution are printed in the
  notebook.
- `RECONSTRUCT_DIR` — per-page reconstruction PDFs (`None` to skip).

Both pipelines run real PaddleOCR, and the archive pipeline also shells out to
LibreOffice, so keep `PAGES_PER_PDF` small.


In [ ]:
# Directory tree searched recursively for .pdf + label .json files.
DATASET_ROOT = PROJECT_ROOT / "references"

# Page cap for PDFs in the tree that have no sidecar label file (auto labels
# only). PDFs a label file references use exactly the pages it names.
PAGES_PER_PDF = 3

# Per-page evaluation reports + per-page stage timing go here (current) /
# RESULTS_TXT.with_name(stem + "_legacy.txt") (legacy). Only aggregates print.
RESULTS_TXT = PROJECT_ROOT / "benchmark_results.txt"

# Where per-page reconstruction PDFs are written; None disables PDF output.
RECONSTRUCT_DIR = PROJECT_ROOT / "benchmark_reconstructions"

# How many current-pipeline PaddleOCR cluster renders to showcase (split
# ~50/50 between OCR passes and blank OCR failures).
SHOWCASE_N = 20
SHOWCASE_SEED = 0

IOU_THRESHOLD = DEFAULT_IOU_THRESHOLD

# Archive's raster-fallback stage shells out to LibreOffice (`soffice`) to
# strip native content before its Type-4 OCR/autotrace pass -- set True only
# if LibreOffice is installed and on PATH; otherwise this stays Type-2-only
# (native + fill-vector OCR), which is still a fair comparison since the
# current pipeline being benchmarked has no raster-image stage either.
ENABLE_ARCHIVE_RASTER_PASS = False

In [ ]:
# Recursively collect every PDF + every label file under DATASET_ROOT into one
# flat (pdf, page) dataset (see rastervec.Reader.dataset.collect_dataset).
dataset = collect_dataset(DATASET_ROOT, pages_per_pdf=PAGES_PER_PDF)

pdf_pages = [(d.pdf_path, d.page_index) for d in dataset]
# manual_entries[(pdf_path, page_index)] -> list[LabelEntry] (source == "manual")
manual_entries = {
    (d.pdf_path, d.page_index): list(d.manual_entries) for d in dataset
}

n_manual = sum(len(v) for v in manual_entries.values())
n_labelled_pages = sum(1 for v in manual_entries.values() if v)
print(f"{len(pdf_pages)} (pdf, page) pair(s) under {DATASET_ROOT}; "
      f"{n_manual} manual label(s) across {n_labelled_pages} page(s)")

In [ ]:
def prepare_page(pdf_path: str, page_index: int) -> dict:
    """One shared prep step per (pdf, page), run once ahead of both pipeline
    loops below: ground truth labels (auto + any loaded manual) AND the
    native-text -> vector-drawn-text Conversion the current pipeline needs --
    computed together so both the current and legacy loops score against,
    and the current pipeline runs against, exactly the same precomputed
    data (no per-loop recompute of ground truth, no per-run reconversion)."""
    labels = auto_label_pdf(pdf_path, page_index)  # source="auto"
    labels.entries.extend(manual_entries.get((pdf_path, page_index), []))  # source="manual"
    converted_bytes = convert_page_to_vector_text(pdf_path, page_index)
    return {"gt": labels, "converted_bytes": converted_bytes}


def run_current_page(converted_bytes: bytes) -> dict:
    """Real current-pipeline run (OCR included) on the shared prep step's
    already-converted bytes -> the ctx fields evaluate_pipeline /
    reconstruction need. Conversion matters: the current Vector_Classification
    + OCR chain only ever scores vector-path-drawn text, so on an unconverted
    native PDF every label would come back "not_found".

    `stage_durations` is ctx.stage_durations -- wall-clock seconds per pipeline
    stage (see pipeline.Pipeline._run_stages); its sum is the per-page total."""
    with tempfile.TemporaryDirectory() as tmp_dir:
        converted_path = str(Path(tmp_dir) / "converted.pdf")
        Path(converted_path).write_bytes(converted_bytes)
        with Reader(converted_path) as reader:
            ctx = run_page_context(reader, 0)
    return {
        "cluster_ocr_results": ctx.cluster_ocr_results or [],
        "ocr_results": ctx.ocr_results or [],
        "drawing_vectors": ctx.drawing_vectors or [],
        "clustering": ctx.clustering,
        "fast_dropped": ctx.fast_dropped,
        "ocr_failed": ctx.ocr_failed,
        "stage_durations": dict(ctx.stage_durations or {}),
    }


def run_legacy_page(pdf_path: str, page_index: int) -> dict:
    """Archive's own pipeline, run unmodified on the ORIGINAL native PDF via
    legacy_adapter (no Conversion -- archive reads native text directly).
    Reshaped into the same dict shape as run_current_page; the loss-funnel
    inputs (clustering/fast_dropped/ocr_failed) don't exist for archive."""
    elements = run_archive_pipeline(
        pdf_path, page_index, enable_raster_pass=ENABLE_ARCHIVE_RASTER_PASS,
    )
    cluster_ocr_results = to_cluster_ocr_results(elements, page_index=page_index)
    return {
        "cluster_ocr_results": cluster_ocr_results,
        "ocr_results": [c.resolved for c in cluster_ocr_results],
        "drawing_vectors": to_drawing_vectors(elements, page_index=page_index),
        "clustering": None,
        "fast_dropped": None,
        "ocr_failed": None,
    }


def evaluate_outputs(outputs: dict, labels, iou_threshold: float):
    return evaluate_pipeline(
        labels,
        outputs["cluster_ocr_results"],
        outputs["drawing_vectors"],
        iou_threshold=iou_threshold,
        clustering=outputs["clustering"],
        fast_dropped=outputs["fast_dropped"],
        ocr_failed=outputs["ocr_failed"],
    )


def format_page_timing(durations: dict) -> str:
    """One-line per-stage + total wall-clock, appended after a page's
    format_report block in the results .txt file."""
    if not durations:
        return "  stage timing: (none)"
    parts = "  ".join(f"{k}={v:.2f}s" for k, v in durations.items())
    return f"  stage timing: {parts}  |  total={sum(durations.values()):.2f}s"


def _page_meta(pdf_path: str, page_index: int):
    with Reader(pdf_path) as reader:
        return reader.get_page(page_index).meta


def write_reconstructions(pdf_path, page_index, gt, current_outputs, legacy_outputs):
    """Text-found (+ drawing vectors) redrawn as real PDFs for visual
    comparison. groundtruth + current share the current pipeline's drawing
    layer so they overlay directly; legacy is text-only (its DrawingVectors
    carry no path geometry)."""
    if not RECONSTRUCT_DIR:
        return
    RECONSTRUCT_DIR.mkdir(parents=True, exist_ok=True)
    page_meta = _page_meta(pdf_path, page_index)
    stem = Path(pdf_path).stem
    drawings = current_outputs["drawing_vectors"] if current_outputs else None

    (RECONSTRUCT_DIR / f"{stem}_p{page_index}_groundtruth.pdf").write_bytes(
        render_reconstructed_pdf(
            page_meta,
            text_boxes=[(e.text, e.cluster_bbox, e.expected_rotation) for e in gt.entries],
            drawing_vectors=drawings,
        )
    )
    if current_outputs:
        (RECONSTRUCT_DIR / f"{stem}_p{page_index}_current.pdf").write_bytes(
            render_reconstructed_pdf(
                page_meta,
                ocr_results=current_outputs["ocr_results"],
                drawing_vectors=current_outputs["drawing_vectors"],
            )
        )
    if legacy_outputs:
        (RECONSTRUCT_DIR / f"{stem}_p{page_index}_legacy.pdf").write_bytes(
            render_reconstructed_pdf(page_meta, ocr_results=legacy_outputs["ocr_results"])
        )


def render_ocr_input(cluster, dpi: int = 300):
    """The exact image `RenderOCR.ocr_cluster` feeds PaddleOCR for this
    cluster -- `render_vector_cluster` at the same dpi, bumped upward the
    same way so a tiny cluster isn't starved of resolution."""
    width_pt, height_pt = cluster_frame_size(cluster)
    min_side_pt = min(width_pt, height_pt)
    if min_side_pt > 0:
        dpi = max(dpi, math.ceil(MIN_RENDER_SIDE_PX * 72.0 / min_side_pt))
    return render_vector_cluster(cluster, dpi)

In [ ]:
# Ground truth + Conversion, computed once per (pdf, page) ahead of both
# pipeline loops below -- see prepare_page's own docstring.
page_prep = {
    key: prepare_page(*key)
    for key in tqdm(pdf_pages, desc="ground truth + conversion")
}

## Run the pipelines

The current (new) and legacy (old) runs are separate cells so you can run
either on its own. Each produces two result lists — `*_auto_results` and
`*_manual_results` (the latter stays empty unless a sidecar `.json` supplied
manual labels for a page). The aggregate + chart cells below tolerate any of
the four lists being empty. Per-page reports are written to the results `.txt`
files, not printed.

### Current (new) pipeline

In [ ]:
current_auto_results = []
current_manual_results = []
current_outputs_by_page = {}  # reused by the legacy cell for the shared drawing layer
current_ocr_pool = []         # every ClusterOcrResult, for the PaddleOCR render showcase
current_stage_timings = []    # list[dict[stage_key -> seconds]], one per page
_report_lines = []            # per-page format_report blocks -> RESULTS_TXT (not printed)

for pdf_path, page_index in tqdm(pdf_pages, desc="current"):
    try:
        prep = page_prep[(pdf_path, page_index)]
        outputs = run_current_page(prep["converted_bytes"])
        current_outputs_by_page[(pdf_path, page_index)] = outputs
        current_ocr_pool.extend(outputs["cluster_ocr_results"])
        current_stage_timings.append(outputs["stage_durations"])

        gt = prep["gt"]
        by_src = split_labelset_by_source(gt)

        r_auto = evaluate_outputs(outputs, by_src["auto"], IOU_THRESHOLD)
        current_auto_results.append(r_auto)
        _report_lines.append(format_report(f"[current/auto]   {pdf_path}", page_index, r_auto))

        if by_src["manual"].entries:
            r_manual = evaluate_outputs(outputs, by_src["manual"], IOU_THRESHOLD)
            current_manual_results.append(r_manual)
            _report_lines.append(format_report(f"[current/manual] {pdf_path}", page_index, r_manual))

        _report_lines.append(format_page_timing(outputs["stage_durations"]))

        write_reconstructions(pdf_path, page_index, gt, outputs, None)
    except Exception as exc:  # noqa: BLE001 -- keep benchmarking the rest of the dataset
        _report_lines.append(f"[current] {pdf_path} page {page_index} failed: {exc}")

RESULTS_TXT.write_text("\n\n".join(_report_lines) + "\n", encoding="utf-8")
print(f"per-page reports -> {RESULTS_TXT}  "
      f"({len(current_auto_results)} auto, {len(current_manual_results)} manual page-scores)")

### PaddleOCR render showcase (current pipeline)

`SHOWCASE_N` of the cluster images the current pipeline actually fed to
PaddleOCR (`render_ocr_input` == `RenderOCR.ocr_cluster`'s own render), sampled
randomly with `SHOWCASE_SEED`, split ~50/50 between **PASS** (non-blank OCR
reading) and **FAIL** (blank). If one side is short, the rest is topped up from
the remaining pool. Needs the current-pipeline cell above to have run.

In [ ]:
import matplotlib.pyplot as plt

pool = globals().get("current_ocr_pool", [])
passed = [r for r in pool if r.resolved.text.strip()]
failed = [r for r in pool if not r.resolved.text.strip()]
print(f"pool: {len(pool)} cluster OCR results  ({len(passed)} passed, {len(failed)} blank)")

rng = random.Random(SHOWCASE_SEED)
half = SHOWCASE_N // 2
pick_pass = rng.sample(passed, min(half, len(passed)))
pick_fail = rng.sample(failed, min(SHOWCASE_N - len(pick_pass), len(failed)))
# top the shorter side up from whatever's left so we still show ~SHOWCASE_N
chosen_ids = {id(r) for r in pick_pass + pick_fail}
remaining = [r for r in pool if id(r) not in chosen_ids]
rng.shuffle(remaining)
pick = pick_pass + pick_fail + remaining[: max(0, SHOWCASE_N - len(chosen_ids))]
rng.shuffle(pick)

if not pick:
    print("nothing to showcase -- run the current-pipeline cell first")
else:
    cols = 4
    rows = math.ceil(len(pick) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.0), squeeze=False)
    for ax, result in zip(axes.flat, pick):
        ax.imshow(render_ocr_input(result.cluster))
        text = result.resolved.text.strip()
        ok = bool(text)
        label = f'PASS  "{text}"' if ok else "FAIL  (blank OCR)"
        ax.set_title(label[:46], color="#1a7f37" if ok else "#cf222e", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.flat[len(pick):]:
        ax.axis("off")
    n_pass = sum(1 for r in pick if r.resolved.text.strip())
    fig.suptitle(
        f"Current-pipeline PaddleOCR cluster renders "
        f"({n_pass} pass / {len(pick) - n_pass} fail)",
        y=1.0,
    )
    plt.tight_layout()
    plt.show()

### Legacy (old / archive) pipeline

Independent of the current-pipeline cell above — run it only if you have the
archive tree at `<repo_root>/archive/` (it is not checked into this repo).
Without it, `run_legacy_page` raises `ModuleNotFoundError: raster_parser` per
page and the loop just logs and moves on, leaving both legacy result lists
empty. If the current cell ran first, this cell also writes the `_current.pdf`
reconstruction alongside `_legacy.pdf` (same drawing layer).

In [ ]:
legacy_auto_results = []
legacy_manual_results = []
legacy_page_totals = []  # per-page wall-clock of the whole archive extract() call
_current_outputs_by_page = globals().get("current_outputs_by_page", {})
_legacy_txt = RESULTS_TXT.with_name(RESULTS_TXT.stem + "_legacy.txt")
_legacy_report_lines = []

for pdf_path, page_index in tqdm(pdf_pages, desc="legacy"):
    try:
        _t0 = time.perf_counter()
        outputs = run_legacy_page(pdf_path, page_index)
        _elapsed = time.perf_counter() - _t0
        legacy_page_totals.append(_elapsed)

        gt = page_prep[(pdf_path, page_index)]["gt"]
        by_src = split_labelset_by_source(gt)

        r_auto = evaluate_outputs(outputs, by_src["auto"], IOU_THRESHOLD)
        legacy_auto_results.append(r_auto)
        _legacy_report_lines.append(format_report(f"[legacy/auto]   {pdf_path}", page_index, r_auto))

        if by_src["manual"].entries:
            r_manual = evaluate_outputs(outputs, by_src["manual"], IOU_THRESHOLD)
            legacy_manual_results.append(r_manual)
            _legacy_report_lines.append(format_report(f"[legacy/manual] {pdf_path}", page_index, r_manual))

        _legacy_report_lines.append(f"  total time: {_elapsed:.2f}s")

        write_reconstructions(
            pdf_path, page_index, gt,
            _current_outputs_by_page.get((pdf_path, page_index)), outputs,
        )
    except Exception as exc:  # noqa: BLE001
        _legacy_report_lines.append(f"[legacy] {pdf_path} page {page_index} failed: {exc}")

_legacy_txt.write_text("\n\n".join(_legacy_report_lines) + "\n", encoding="utf-8")
print(f"per-page legacy reports -> {_legacy_txt}  "
      f"({len(legacy_auto_results)} auto, {len(legacy_manual_results)} manual page-scores)")

In [ ]:
def print_aggregate(name, results):
    agg = aggregate_results(results)
    print(f"{name}  (n={len(results)}):")
    if not agg:
        print("  (no results)")
    for key, value in agg.items():
        print(f"  {key}: {value}")
    print()
    return agg


current_auto_agg = print_aggregate("current / auto", globals().get("current_auto_results", []))
current_manual_agg = print_aggregate("current / manual", globals().get("current_manual_results", []))
legacy_auto_agg = print_aggregate("legacy / auto", globals().get("legacy_auto_results", []))
legacy_manual_agg = print_aggregate("legacy / manual", globals().get("legacy_manual_results", []))

# --- timing distribution (min / Q1 / median / mean / Q3 / max) ---
timing_summary = summarize_stage_timings(
    globals().get("current_stage_timings", []), Pipeline.stage_keys(),
)
timing_report = format_timing_report(
    timing_summary, title="Current pipeline -- per-stage timing (seconds)",
)
print(timing_report)

legacy_totals = globals().get("legacy_page_totals", [])
legacy_timing_line = ""
if legacy_totals:
    lt = distribution_stats(legacy_totals)
    legacy_timing_line = (
        f"\nLegacy pipeline -- per-page total (seconds): n={lt['n']}  "
        f"min={lt['min']:.2f}  q1={lt['q1']:.2f}  median={lt['median']:.2f}  "
        f"mean={lt['mean']:.2f}  q3={lt['q3']:.2f}  max={lt['max']:.2f}"
    )
    print(legacy_timing_line)

# Append the timing tables to the current-pipeline results .txt file.
if globals().get("current_stage_timings") and RESULTS_TXT.exists():
    with RESULTS_TXT.open("a", encoding="utf-8") as fh:
        fh.write("\n\n" + timing_report + legacy_timing_line + "\n")

## Reading the results

- The **new** pipeline (`current/*`) is expected to match or beat the **old**
  archive pipeline (`legacy/*`) on every metric. A page where the old pipeline
  is ahead is a concrete regression — open that page's block in `RESULTS_TXT`
  and check its `miss reasons` counts for where the new pipeline lost the text.
- **auto vs. manual**: auto ground truth is every native-text line; manual
  ground truth is only the clusters a human labelled in `manual_label.py`.
  Manual is usually the stricter, more curated set — a big gap between
  `*/auto` and `*/manual` on the same pipeline points at auto labels that don't
  correspond to a real recoverable text cluster (or vice-versa).
- **Timing**: the aggregate cell prints the per-stage and per-page-total
  distribution (min / Q1 / median / mean / Q3 / max); per-page stage timing is
  in `RESULTS_TXT`. `ocr_compare` and `fast_text_detect` dominate — a jump in
  either stage's max vs. median points at a pathological page.
- **Reconstruction PDFs** in `RECONSTRUCT_DIR`: open `_groundtruth.pdf` next to
  `_current.pdf` (same page size, same drawing layer) and check whether the
  found text sits where the ground-truth text is. `_legacy.pdf` is text-only.
- **PaddleOCR render showcase**: the FAIL tiles are the clusters that reached
  OCR but came back blank (folded into `drawing_vectors`) — scan them for
  renders that clearly *are* text the OCR should have read (a rendering bug, a
  bad crop, an under-/over-merged cluster) vs. genuine non-text.


In [ ]:
import matplotlib.pyplot as plt

_METRICS = (
    "characters_found_pct",
    "character_accuracy",
    "rotation_accuracy",
    "bbox_accuracy",
    "classification_precision",
    "classification_recall",
)

_candidate_series = [
    ("current/auto", globals().get("current_auto_agg", {})),
    ("current/manual", globals().get("current_manual_agg", {})),
    ("legacy/auto", globals().get("legacy_auto_agg", {})),
    ("legacy/manual", globals().get("legacy_manual_agg", {})),
]
series = [(name, agg) for name, agg in _candidate_series if agg]

x = range(len(_METRICS))
n = max(len(series), 1)
width = 0.8 / n
fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, agg) in enumerate(series):
    offset = (i - (n - 1) / 2) * width
    ax.bar([j + offset for j in x], [agg.get(m, 0.0) for m in _METRICS], width, label=name)
ax.set_xticks(list(x))
ax.set_xticklabels(_METRICS, rotation=30, ha="right")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("New vs. old vector classification -- auto vs. manual ground truth")
plt.tight_layout()
plt.show()

## Files written

- `RESULTS_TXT` — per-page `format_report` blocks + per-page stage timing for
  the current pipeline, with the aggregated timing tables appended at the end.
- `RESULTS_TXT.with_name(<stem>_legacy.txt)` — the same for the legacy pipeline
  (per-page total time only).
- `RECONSTRUCT_DIR/<stem>_p<N>_{groundtruth,current,legacy}.pdf` — found text
  (+ drawing vectors) redrawn as real PDFs, when `RECONSTRUCT_DIR` is set.


In [ ]:
import matplotlib.pyplot as plt

# Per-stage median wall-clock (current pipeline), with min-max whiskers.
_summary = globals().get("timing_summary", {})
_stages = [k for k in _summary if k != "total"]
if not _stages:
    print("no timing data -- run the current-pipeline cell first")
else:
    medians = [_summary[k]["median"] for k in _stages]
    lo = [_summary[k]["median"] - _summary[k]["min"] for k in _stages]
    hi = [_summary[k]["max"] - _summary[k]["median"] for k in _stages]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(range(len(_stages)), medians, yerr=[lo, hi], capsize=3, color="#4c78a8")
    ax.set_xticks(range(len(_stages)))
    ax.set_xticklabels(_stages, rotation=30, ha="right")
    ax.set_ylabel("seconds")
    total = _summary["total"]
    ax.set_title(
        f"Current pipeline per-stage time (median, min-max whiskers)  "
        f"|  per-page total median {total['median']:.1f}s "
        f"(min {total['min']:.1f}s / max {total['max']:.1f}s)"
    )
    plt.tight_layout()
    plt.show()

## Reading the results

- The **new** pipeline (`current/*`) is expected to match or beat the **old**
  archive pipeline (`legacy/*`) on every metric. A page where the old pipeline
  is ahead is a concrete regression — check `miss_reason_counts` in that page's
  `format_report` line for where the new pipeline lost the text.
- **auto vs. manual**: auto ground truth is every native-text line; manual
  ground truth is only the clusters a human labelled in `manual_label.py`.
  Manual is usually the stricter, more curated set — a big gap between
  `*/auto` and `*/manual` on the same pipeline points at auto labels that don't
  correspond to a real recoverable text cluster (or vice-versa).
- **Reconstruction PDFs** in `RECONSTRUCT_DIR`: open `_groundtruth.pdf` next to
  `_current.pdf` (same page size, same drawing layer) and check whether the
  found text sits where the ground-truth text is. `_legacy.pdf` is text-only.
